# NB4: Stage 2 SemEval 2014 (Joint FAISS Index)

Trong Notebook này, chúng ta gộp dữ liệu SemEval 2014 và MAMS để tạo ra một tập FAISS Index lớn hơn (để bổ sung lượng mẫu Neutral dồi dào từ MAMS). Sau đó train lại mô hình Aux Loss trên SemEval.

## 0. Setup & Clone Repo

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml
import os, sys, json, shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

## 1. Wire Datasets & Prepare Data

In [ ]:
# --- SemEval 2014 ---
KAGGLE_SEMEVAL = None
for candidate in ['/kaggle/input/semeval-2014-absa-restaurant',
                  '/kaggle/input/datasets/lcminhc/semeval-2014-absa-restaurant',
                  '/kaggle/input/datasets/duclm318/semeval-2014-absa-restaurant']:
    if os.path.exists(candidate):
        KAGGLE_SEMEVAL = candidate
        break
assert KAGGLE_SEMEVAL, 'Dataset semeval-2014-absa-restaurant not found'

os.makedirs('SemEval-2014', exist_ok=True)
shutil.copy(f'{KAGGLE_SEMEVAL}/Restaurants_Train.xml', 'SemEval-2014/Restaurants_Train.xml')
shutil.copy(f'{KAGGLE_SEMEVAL}/Restaurants_Test_Gold.xml', 'SemEval-2014/Restaurants_Test_Gold.xml')

# --- MAMS ---
if not os.path.exists('data/mams'):
    !git clone --depth 1 https://github.com/siat-nlp/MAMS-for-ABSA.git data/mams

# --- Prepare data ---
!python scripts/01_prepare_data.py
!python scripts/01_prepare_data_mams.py --base_dir . --out_dir data/processed_mams

# --- Wire p5-embed-v4 ---
EMB = None
for candidate in ['/kaggle/input/p5-embed-v4',
                  '/kaggle/input/datasets/lcminhc/p5-embed-v4',
                  '/kaggle/input/datasets/duclm318/p5-embed-v4']:
    if os.path.exists(candidate):
        EMB = candidate
        break
assert EMB, 'Dataset p5-embed-v4 not found'

os.makedirs('checkpoints/embedding_2014', exist_ok=True)
shutil.copy(f'{EMB}/embedding_v4_s2_best.pt', 'checkpoints/embedding_2014/best.pt')


## 2. Merge Sentiment Records & Build Joint FAISS Index

In [ ]:
# Gộp sentiment records của SemEval và MAMS lại với nhau
import json

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

sem_records = load_jsonl('data/processed/sentiment_records.jsonl')
mams_records = load_jsonl('data/processed_mams/sentiment_records.jsonl')

# Để tránh trùng lặp ID (nếu có), ta có thể thêm tiền tố vào ID của MAMS
for r in mams_records:
    r['id'] = 'mams_' + str(r['id'])

joint_records = sem_records + mams_records

with open('data/processed/joint_sentiment_records.jsonl', 'w', encoding='utf-8') as f:
    for r in joint_records:
        f.write(json.dumps(r) + '\n')

print(f"SemEval: {len(sem_records)} | MAMS: {len(mams_records)} | Joint: {len(joint_records)}")

# Build FAISS index từ file joint
os.makedirs('indexes/joint', exist_ok=True)
!python scripts/03_build_index.py \
    --embedding_ckpt checkpoints/embedding_2014/best.pt \
    --input data/processed/joint_sentiment_records.jsonl \
    --out_dir indexes/joint/


## 3. Train Stage 2 (Aux Loss) on SemEval with Joint Index

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

# Thay vì train cả No-retrieval, ta chỉ cần train Aux Loss vì No-Retrieval không liên quan tới index
!python scripts/04b_train_stage2.py \
    --config configs/stage2_2014_auxloss.yaml \
    --embedding_ckpt checkpoints/embedding_2014/best.pt \
    --index_dir indexes/joint/ \
    --retrieval_config configs/retrieval_v2.yaml


## 4. Evaluate Gold-Category Accuracy

In [ ]:
import subprocess

EVAL_EXPERIMENTS = [
    {"name": "Aux Loss (Joint Index)", "ckpt_dir": "checkpoints/stage2_2014_auxloss", "config": "configs/stage2_2014_auxloss.yaml", "no_retrieval": False},
]

print("=" * 60)
print("GOLD-CATEGORY ACCURACY (SemEval 2014 Test Set)")
print("=" * 60)

for exp in EVAL_EXPERIMENTS:
    if not os.path.exists(f'{exp["ckpt_dir"]}/best.pt'):
        print(f'SKIP {exp["name"]} — checkpoint missing'); continue
        
    cmd = ["python", "scripts/06_evaluate_sentiment_only.py",
           "--stage2_ckpt", f'{exp["ckpt_dir"]}/best.pt',
           "--stage2_config", exp["config"],
           "--data_dir", "data/processed"]
    if exp["no_retrieval"]:
        cmd.append("--no_retrieval")
    else:
        cmd += ["--embedding_ckpt", "checkpoints/embedding_2014/best.pt", "--index_dir", "indexes/joint/"]
    
    print(f'\n--- {exp["name"]} ---')
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("ERROR:")
        print(result.stderr)
    else:
        print(result.stdout)


## 5. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_nb4_joint'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f'{output_dir}/logs', exist_ok=True)

for name in ['stage2_2014_auxloss']:
    src = f'checkpoints/{name}/best.pt'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/{name}_best.pt')
        print(f'{name}_best.pt saved.')
        
    log_src = f'logs/{name}_training.jsonl'
    if os.path.exists(log_src):
        shutil.copy(log_src, f'{output_dir}/logs/')

shutil.make_archive('/kaggle/working/outputs_nb4_joint_backup', 'zip', '/kaggle/working', 'outputs_nb4_joint')
print("Saved outputs successfully!")
